# Alpha decay and information half-life (Prompt #8, Steps 31-34)

IC by horizon on cumulative targets (dilution ~ 1/sqrt(h)) and on marginal targets (the single bar h ahead), the peak and half-decay horizons, and an information half-life fitted only where the marginal curve is exponential. `xq alpha-decay --timeframe 1h --feature log_rv_20` prints one feature.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.features` / `xauusd_quant.alpha` / `xauusd_quant.research.feature_*` and read back from `results/feature_research/`. Generate those first:

```
xq build-feature-factory --timeframe 1h      # versioned causal matrix + targets
xq feature-research --timeframe 1h           # every stage, statuses, ledger, figures
xq feature-research --all                    # 5m, 15m, 30m, 1h (one child process each)
xq feature-research --compare                # cross-timeframe tables
```

**Descriptive research only.** Statuses (`strong_candidate` ... `failed_null`) are statistical information about rank ICs against circular-shift, pipeline and noise-feature nulls - not signals, not a model, no cost, no PnL. Features at `t` use bars `<= t`; targets live in their own namespace (`target_*`) and are never joined into the matrix.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.features.factory import current_version_dir
from xauusd_quant.features.factory_config import load_features_config

fcfg = load_features_config(ROOT / "config" / "features.yaml")

TIMEFRAME = "1h"       # 5m | 15m | 30m | 1h - change and re-run

base = fcfg.results_path / TIMEFRAME
if not base.exists():
    print(f"No research for {TIMEFRAME}. Run:  xq feature-research --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated research table (None if it was not written)."""
    path = base / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = base / "figures" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no figure {path.name})")

def load_json(name):
    path = base / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Decay curves of the leading candidates

In [ ]:
plot('ic_decay')

## Decay table

In [ ]:
decay = table('ic/alpha_decay')
decay.filter(pl.col('method') == 'spearman').select('feature', 'target_family', 'curve', 'peak_horizon',
          'half_decay_horizon', 'zero_horizon', 'alpha_information_half_life', 'fit').sort('feature')

## Horizon profile summary

In [ ]:
decay.group_by('target_family', 'curve').agg(pl.col('alpha_information_half_life').median())